# 05 · 학습 epoch 선택 근거: validation loss와 pseudo-mIoU

기존 ACDC fold 0 학습 로그만 읽는 **로컬 CPU 분석**입니다. GPU, PyTorch, nnU-Net 설치, 원본 MRI, 추론 결과는 필요하지 않습니다. 저장소 `docs/artifacts`에 보관한 로그를 기본으로 사용합니다.

분석 항목: validation loss 최솟값·pseudo-mIoU 최댓값과 최솟값, 최고 기록 갱신 위치, 마지막 갱신 이후 경과 epoch, 50 epoch 구간 평균, 후보 250 epoch의 시간 절감 근거.

**중요:** mIoU는 매 epoch 기록된 RV/MLV/LVC pseudo-Dice를 각각 `IoU = Dice / (2 − Dice)`로 변환한 후 평균한 근삿값입니다. 배경 제외, validation 패치 기반이며 전체 3D 볼륨 mIoU가 아닙니다. 로그는 소수 네 자리로 반올림되어 있으므로 동률과 아주 작은 개선의 판정에도 한계가 있습니다. 그래프는 1부터 시작하는 epoch 번호를 사용하고 원본의 0-based 번호도 CSV에 보존합니다.

## 0. 빈 venv에서 시작
이미 만들어 둔 venv를 사용합니다. 기존 Jupyter에서 이 venv를 커널로 선택하려면 **먼저 터미널에서** 다음을 실행하세요(경로는 자신의 venv에 맞게 변경).

```bash
source /path/to/venv/bin/activate
python -m pip install ipykernel
python -m ipykernel install --user --name acdc-analysis --display-name "ACDC analysis"
```

이후 Jupyter에서 **ACDC analysis** 커널을 선택합니다. 아직 Jupyter를 실행할 환경도 없다면 venv에서 `python -m pip install jupyterlab` 후 `python -m jupyterlab`을 실행하세요. 분석 패키지는 아래 셀이 설치합니다. Python 3.11 또는 3.12를 권장합니다.

In [ ]:
import sys, subprocess
sys.dont_write_bytecode = True
INSTALL_DEPENDENCIES = True  # 설치 완료 후 재실행할 때는 False로 변경 가능
print('Kernel Python:', sys.executable)
if INSTALL_DEPENDENCIES:
    if sys.prefix == sys.base_prefix:
        raise RuntimeError('시스템 Python입니다. 먼저 만들어 둔 venv 커널을 선택하세요.')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-cache-dir',
        'numpy>=1.24,<3', 'pandas>=2,<3', 'matplotlib>=3.8,<4', 'ipykernel>=6,<7'
    ])
    print('분석 의존성 설치 완료. 이미 numpy/pandas를 불러온 커널에서 설치했다면 재시작하세요.')

In [ ]:
from pathlib import Path
import re, json, hashlib, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'experiments/acdc').is_dir()), None)
if REPO is None:
    raise RuntimeError('nnUNet 저장소 안에서 이 노트북을 실행하세요.')
DOCS = REPO / 'experiments/acdc/docs'
LOG_DIR = (DOCS / 'artifacts/runs/acdc-baseline-v1/nnUNet_results/Dataset027_ACDC'
           / 'nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0')
LOG_FILES = sorted(LOG_DIR.glob('training_log_*.txt'))
# 다른 로그 분석 시 LOG_FILES를 경로 목록으로 직접 지정하세요.
# 재시작 로그가 여러 개면 오래된 로그 → 최신 로그 순서로 지정합니다.
OUTPUT_DIR = DOCS / 'epoch-selection'
CANDIDATE_EPOCH = 250
WINDOW = 50
ZOOM_START = 50  # 초기 급격한 변화 이후 확대 그래프
SAVE_ARTIFACTS = True  # PNG/PDF/CSV/JSON/Markdown을 OUTPUT_DIR에 저장
if not LOG_FILES:
    raise FileNotFoundError(f'학습 로그를 찾을 수 없습니다: {LOG_DIR}')
if WINDOW < 2 or CANDIDATE_EPOCH < 1:
    raise ValueError('WINDOW >= 2, CANDIDATE_EPOCH >= 1 이어야 합니다.')
print('Logs:', *LOG_FILES, sep='\n')
print('Outputs:', OUTPUT_DIR)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,
                     'pdf.fonttype': 42, 'savefig.dpi': 180})

## 1. 로그 읽기와 입력 검증
한 epoch의 train loss, validation loss, 클래스별 pseudo-Dice, 소요 시간이 모두 기록된 경우만 사용합니다. 재시작으로 같은 epoch가 반복되면 **마지막으로 완료된 기록**을 사용합니다. 미완료 epoch는 제외하고 수를 표시하며, 완료 epoch가 0부터 연속되어 있는지 확인합니다. 로그를 Python 코드로 실행하지 않습니다.

In [ ]:
NUMBER = r'[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?'

def parse_logs(files):
    records, incomplete = {}, []
    required = {'epoch_index', 'train_loss', 'val_loss', 'dice_RV', 'dice_MLV', 'dice_LVC', 'epoch_seconds'}
    for path in files:
        current = None
        def finish(row):
            if row is not None:
                if required <= row.keys():
                    records[row['epoch_index']] = row.copy()
                else:
                    incomplete.append((str(path), row['epoch_index']))
        for line in Path(path).read_text().splitlines():
            start = re.search(r': Epoch (\d+)\s*$', line)
            if start:
                finish(current)
                current = {'epoch_index': int(start[1]), 'source_log': Path(path).name}
                continue
            if current is None:
                continue
            for key, token in [('train_loss', 'train_loss'), ('val_loss', 'val_loss'),
                               ('epoch_seconds', 'Epoch time:')]:
                match = re.search(re.escape(token) + r'\s+(' + NUMBER + ')', line)
                if match:
                    current[key] = float(match[1])
            if 'Pseudo dice [' in line:
                payload = line.split('Pseudo dice [', 1)[1].split(']', 1)[0]
                payload = re.sub(r'(?:np\.)?float(?:16|32|64)\(([^()]*)\)', r'\1', payload)
                values = [float(item.strip()) for item in payload.split(',')]
                if len(values) != 3 or not all(np.isfinite(values)) or not all(0 <= x <= 1 for x in values):
                    raise ValueError(f'Expected finite RV/MLV/LVC Dice in [0,1]: {line}')
                current.update(dict(zip(['dice_RV', 'dice_MLV', 'dice_LVC'], values)))
        finish(current)
    if not records:
        raise ValueError('완료된 epoch가 없습니다.')
    frame = pd.DataFrame(records.values()).sort_values('epoch_index').reset_index(drop=True)
    if frame.epoch_index.tolist() != list(range(int(frame.epoch_index.max()) + 1)):
        raise ValueError('로그의 epoch가 0부터 연속되어 있지 않습니다. 빠진 로그를 추가하세요.')
    numeric = frame.select_dtypes(include='number')
    if not np.isfinite(numeric.to_numpy()).all() or (frame.epoch_seconds < 0).any():
        raise ValueError('Non-finite metrics or invalid duration')
    print('Incomplete epoch blocks ignored:', len(incomplete))
    return frame

df = parse_logs(LOG_FILES)
df['epoch'] = df.epoch_index + 1
for region in ['RV', 'MLV', 'LVC']:
    df[f'iou_{region}'] = df[f'dice_{region}'] / (2 - df[f'dice_{region}'])
df['pseudo_miou_pct'] = df[['iou_RV', 'iou_MLV', 'iou_LVC']].mean(axis=1) * 100
assert CANDIDATE_EPOCH <= len(df), '후보 epoch가 로그 길이를 초과합니다.'
assert len(df) >= WINDOW, '이동평균 window보다 로그가 짧습니다.'
print(f'{len(df)} complete epochs; display epoch 1..{df.epoch.max()}')
display(df.head())

## 2. 최고 기록 갱신과 극값
`<` / `>`를 사용해 **엄격히 좋아진 경우만 갱신**으로 표시합니다. 동률은 갱신하지 않으며, 최고값이 여러 번 나오면 최초 도달 epoch를 표시합니다. 첫 epoch는 초기 기록입니다.

이동평균은 현재 epoch까지의 **과거 WINDOW개**만 사용하며, 미래 값을 참조하지 않습니다. 전체 최솟값/최댓값은 이동평균이 아닌 원래 로그값으로 정합니다. mIoU 최솟값은 참고로 표시하며 종료 기준은 최댓값입니다.

In [ ]:
df['best_val_loss'] = df.val_loss.cummin()
df['best_pseudo_miou_pct'] = df.pseudo_miou_pct.cummax()
df['loss_record'] = df.val_loss < df.best_val_loss.shift(1, fill_value=np.inf)
df['miou_record'] = df.pseudo_miou_pct > df.best_pseudo_miou_pct.shift(1, fill_value=-np.inf)
for name, flag in [('loss', 'loss_record'), ('miou', 'miou_record')]:
    last_epoch = df.epoch.where(df[flag]).ffill()
    df[f'epochs_since_{name}_record'] = (df.epoch - last_epoch).astype(int)
df['val_loss_mean'] = df.val_loss.rolling(WINDOW, min_periods=WINDOW).mean()
df['pseudo_miou_mean_pct'] = df.pseudo_miou_pct.rolling(WINDOW, min_periods=WINDOW).mean()
best_loss = df.loc[df.val_loss.idxmin()]
best_miou = df.loc[df.pseudo_miou_pct.idxmax()]
worst_miou = df.loc[df.pseudo_miou_pct.idxmin()]
extrema = pd.DataFrame([
    {'metric': 'minimum validation loss', 'epoch': int(best_loss.epoch), 'epoch_index': int(best_loss.epoch_index), 'value': best_loss.val_loss},
    {'metric': 'maximum pseudo-mIoU (%)', 'epoch': int(best_miou.epoch), 'epoch_index': int(best_miou.epoch_index), 'value': best_miou.pseudo_miou_pct},
    {'metric': 'minimum pseudo-mIoU (%) [reference only]', 'epoch': int(worst_miou.epoch), 'epoch_index': int(worst_miou.epoch_index), 'value': worst_miou.pseudo_miou_pct},
])
display(extrema)
updates = df.loc[df.loss_record | df.miou_record,
                 ['epoch', 'epoch_index', 'val_loss', 'pseudo_miou_pct', 'loss_record', 'miou_record']]
print('Record updates (last 15):')
display(updates.tail(15))
print('Epochs since last record at end:',
      df[['epochs_since_loss_record', 'epochs_since_miou_record']].iloc[-1].to_dict())

## 3. 전체 곡선과 최고 기록 갱신 위치
옅은 선=epoch별 값, 진한 선=과거 구간 평균, 계단선=누적 최고 기록, 작은 점=기록 갱신, 별=전역 최적값. 보라색 세로선은 후보 학습 예산입니다. 확대 그림도 같은 데이터를 사용하며 첫 ZOOM_START 이전 구간만 제외합니다.

In [ ]:
def save_figure(fig, stem):
    if SAVE_ARTIFACTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        for extension in ['png', 'pdf']:
            fig.savefig(OUTPUT_DIR / f'{stem}.{extension}', bbox_inches='tight')

def plot_metrics(start=1, stem='metric-history'):
    view = df[df.epoch >= start]
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
    specs = [
        ('val_loss', 'val_loss_mean', 'best_val_loss', 'loss_record', best_loss,
         'Validation loss (lower is better)', 'tab:blue'),
        ('pseudo_miou_pct', 'pseudo_miou_mean_pct', 'best_pseudo_miou_pct', 'miou_record', best_miou,
         'Foreground pseudo-mIoU (%) (higher is better)', 'tab:green')]
    for ax, (metric, avg, record, flag, best, ylabel, color) in zip(axes, specs):
        ax.plot(view.epoch, view[metric], color=color, alpha=.25, lw=.8, label='Per epoch')
        ax.plot(view.epoch, view[avg], color=color, lw=2, label=f'Trailing {WINDOW}-epoch mean')
        ax.step(view.epoch, view[record], where='post', color='black', lw=1.2, label='Best so far')
        marked = view[view[flag]]
        ax.scatter(marked.epoch, marked[metric], color='darkorange', s=16, zorder=4, label='New record')
        if best.epoch >= start:
            ax.scatter([best.epoch], [best[metric]], marker='*', s=180, color='crimson', zorder=5)
        ax.text(.98, .05, f'Global best: epoch {int(best.epoch)} | {best[metric]:.5f}',
                transform=ax.transAxes, ha='right', va='bottom', color='crimson',
                bbox={'facecolor': 'white', 'alpha': .9, 'edgecolor': 'none'})
        ax.axvline(CANDIDATE_EPOCH, color='purple', ls='--', label=f'Budget: {CANDIDATE_EPOCH}')
        ax.set_ylabel(ylabel)
        ax.grid(alpha=.2)
        ax.legend(loc='upper right', fontsize=8, ncol=2)
    if worst_miou.epoch >= start:
        axes[1].scatter([worst_miou.epoch], [worst_miou.pseudo_miou_pct], marker='v', color='gray', s=70)
        axes[1].annotate(f'Min: ep {int(worst_miou.epoch)}, {worst_miou.pseudo_miou_pct:.2f}%',
                         (worst_miou.epoch, worst_miou.pseudo_miou_pct), xytext=(30, 12),
                         textcoords='offset points', fontsize=9)
    axes[1].set_xlabel('Epoch (1-based; log index + 1)')
    fig.suptitle('ACDC fold 0 | validation patches | retrospectively selected budget', fontsize=13)
    save_figure(fig, stem)
    plt.show()
    plt.close(fig)

plot_metrics()
if 1 < ZOOM_START < len(df):
    plot_metrics(ZOOM_START, 'metric-history-zoom')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
ax.plot(df.epoch, df.epochs_since_loss_record, label='Since validation-loss record', lw=2)
ax.plot(df.epoch, df.epochs_since_miou_record, label='Since pseudo-mIoU record', ls='--', lw=2)
ax.axvline(CANDIDATE_EPOCH, color='purple', ls=':', label=f'Budget: {CANDIDATE_EPOCH}')
for point, title, height in [(best_loss, 'Last loss record', .90), (best_miou, 'Last mIoU record', .78)]:
    ax.text(.02, height, f'{title}: epoch {int(point.epoch)}; no later strict improvement', transform=ax.transAxes)
ax.set(xlabel='Epoch (1-based)', ylabel='Epochs since last strict improvement',
       title='Record-update waiting time (ties do not reset the counter)')
ax.legend(loc='lower right')
ax.grid(alpha=.2)
save_figure(fig, 'epochs-without-improvement')
plt.show()
plt.close(fig)

## 4. 구간 평균과 후보 epoch의 근거
구간 내 표준편차는 epoch 사이의 변동이며 신뢰구간이 아닙니다. epoch별 지표는 독립 표본이 아니므로 유의성 검정으로 해석하지 않습니다. 최고 기록의 마지막 갱신 시점은 전체 로그를 본 **사후 관찰**입니다. 온라인 early stopping 규칙으로 자동 확정하지 않습니다.

In [ ]:
rows = []
for start in range(0, len(df), WINDOW):
    block = df.iloc[start:start + WINDOW]
    rows.append(dict(epoch_start=int(block.epoch.min()), epoch_end=int(block.epoch.max()),
                     n_epochs=len(block), val_loss_mean=block.val_loss.mean(),
                     val_loss_std=block.val_loss.std(), pseudo_miou_mean_pct=block.pseudo_miou_pct.mean(),
                     pseudo_miou_std_pp=block.pseudo_miou_pct.std()))
windows = pd.DataFrame(rows)
display(windows)
candidate = df.loc[df.epoch == CANDIDATE_EPOCH].iloc[0]
used_hours = df.loc[df.epoch <= CANDIDATE_EPOCH, 'epoch_seconds'].sum() / 3600
full_hours = df.epoch_seconds.sum() / 3600
summary = {
    'epochs_observed': len(df), 'candidate_epoch': CANDIDATE_EPOCH,
    'candidate_updates_at_250_steps_per_epoch': CANDIDATE_EPOCH * 250,
    'last_loss_record_epoch': int(best_loss.epoch), 'minimum_val_loss': float(best_loss.val_loss),
    'last_miou_record_epoch': int(best_miou.epoch), 'maximum_pseudo_miou_pct': float(best_miou.pseudo_miou_pct),
    'minimum_pseudo_miou_epoch': int(worst_miou.epoch), 'minimum_pseudo_miou_pct': float(worst_miou.pseudo_miou_pct),
    'loss_record_wait_at_candidate': int(candidate.epochs_since_loss_record),
    'miou_record_wait_at_candidate': int(candidate.epochs_since_miou_record),
    'loss_record_wait_at_end': int(df.epochs_since_loss_record.iloc[-1]),
    'miou_record_wait_at_end': int(df.epochs_since_miou_record.iloc[-1]),
    'recorded_epoch_hours_to_candidate': float(used_hours), 'recorded_epoch_hours_total': float(full_hours),
    'recorded_epoch_time_reduction_percent': float(100 * (1 - used_hours / full_hours)),
    'window': WINDOW,
    'sources': [{'path': str(p.relative_to(REPO)) if p.is_relative_to(REPO) else str(p),
                 'sha256': hashlib.sha256(p.read_bytes()).hexdigest()} for p in LOG_FILES],
    'packages': {p: importlib.metadata.version(p) for p in ['numpy', 'pandas', 'matplotlib']},
    'limitations': ['Single ACDC fold 0 baseline; not a universal convergence budget.',
                    'Rounded patch Dice converted per class to approximate foreground pseudo-mIoU.',
                    'A shorter total epoch schedule changes polynomial learning-rate decay.',
                    'Recorded epoch durations exclude final full-volume inference and setup.',
                    'Historical best metric does not imply that an epoch-specific checkpoint was saved.']
}
display(pd.Series(summary).drop(['sources', 'packages', 'limitations']).to_frame('value'))

In [ ]:
report = f"""# ACDC epoch 선택 근거

## 관찰
- 분석 대상: ACDC fold 0, 완료 {len(df)} epoch. 번호는 1-based입니다.
- Validation loss 최솟값: **{best_loss.val_loss:.4f}**, **{int(best_loss.epoch)}번째 epoch** (로그 index {int(best_loss.epoch_index)}).
- Foreground pseudo-mIoU 최댓값: **{best_miou.pseudo_miou_pct:.4f}%**, **{int(best_miou.epoch)}번째 epoch**.
- Pseudo-mIoU 최솟값(참고): {worst_miou.pseudo_miou_pct:.4f}%, {int(worst_miou.epoch)}번째 epoch.
- 마지막 최고 기록 갱신 이후 학습 종료까지 loss는 **{summary['loss_record_wait_at_end']} epoch**, mIoU는 **{summary['miou_record_wait_at_end']} epoch** 동안 엄격한 개선이 없었습니다.
- 후보 {CANDIDATE_EPOCH} epoch 시점에서는 각각 {summary['loss_record_wait_at_candidate']} / {summary['miou_record_wait_at_candidate']} epoch 동안 갱신이 없었습니다. 이 시점에는 이후 결과를 알 수 없었으므로, 위 종료 시점의 대기 길이를 온라인 판단 근거로 사용하면 안 됩니다.

## 제한된 예산의 후보
공통 예산 후보는 **{CANDIDATE_EPOCH} epoch × 250 step = {CANDIDATE_EPOCH * 250:,}회 업데이트**입니다.
기록된 epoch 시간 합계는 후보까지 약 **{used_hours:.2f}시간**, 전체 약 **{full_hours:.2f}시간**이며 약 **{100 * (1-used_hours/full_hours):.1f}%** 절감에 해당합니다. 최종 추론·평가·준비 시간은 제외합니다.

후보의 타당성은 아래 구간 평균과 최고 기록 갱신 위치를 함께 보고 판단합니다. 그래프가 후보를 자동으로 최적이라고 증명하는 것은 아닙니다.

## 한계와 비교 조건
- 지표는 패치 기반 pseudo-mIoU이며 전체 영상 mIoU가 아닙니다. 클래스별 반올림된 Dice를 각각 IoU로 변환 후 평균했습니다.
- 단일 baseline/fold의 사후 분석입니다. 다른 모델의 수렴 시점이나 250 epoch에서의 최종 볼륨 성능을 보장하지 않습니다.
- 총 epoch를 변경하면 polynomial 학습률 일정도 달라집니다. 짧은 일정의 baseline을 별도로 확인해야 합니다.
- 기준 metric과 checkpoint 선택 정책을 구분하세요. 기존 checkpoint_best는 EMA pseudo-Dice 기준이며, 여기서 계산한 최소 loss/최대 mIoU epoch의 가중치가 보존되어 있다는 뜻은 아닙니다.
- 기존 학습의 split/checkpoint를 변경하거나 재학습하지 않았습니다.

## 그래프
![전체 곡선](metric-history.png)
![마지막 갱신 이후 경과 epoch](epochs-without-improvement.png)

## {WINDOW} epoch 구간 평균
| Epoch | Mean val loss | Mean pseudo-mIoU (%) |
|---|---:|---:|
"""
for row in windows.itertuples():
    report += f'| {row.epoch_start}–{row.epoch_end} | {row.val_loss_mean:.6f} | {row.pseudo_miou_mean_pct:.4f} |\n'
if SAVE_ARTIFACTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_DIR / 'epoch-metrics.csv', index=False)
    updates.to_csv(OUTPUT_DIR / 'record-updates.csv', index=False)
    extrema.to_csv(OUTPUT_DIR / 'extrema.csv', index=False)
    windows.to_csv(OUTPUT_DIR / 'window-summary.csv', index=False)
    (OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n')
    (OUTPUT_DIR / 'README.md').write_text(report)
    print('저장 완료:', OUTPUT_DIR)
display(Markdown(report.split('## 그래프')[0]))